# B实际四相位补齐：Card89用户运行包

Run all直接从GitHub现有`dev/local-joint-state-payload-v1`分支加载源码及保存JSON，无需上传ZIP。
若`/content/SC-SSTW-b-real-four-phase-v1`已有可用源码，则直接使用；本地user_bundle.zip仅作可选副本。
Notebook不内嵌源码/数据大字典，实际下载、导入或文件读取失败会直接报告。

固定用户任务：VAE加载1，新增E6/FFT6、encoder名义66，两钥新Q/票各12；D/新writer/codec/媒体decode/梯度0。
复用Card88普通JSON sidecar：旧E45前10行、280个已存候选归约不重算；6个新E41只补起点62/63/64。
16逻辑相位视图/32两钥表不等于16个新E。每查询137状态；408新候选，raw/centered/min均保留。
完整相位族缺项保持未决，不删掉失败项取峰。没有tau/标定、身份认证、总体FPR或可信完整32位。

首cell只mount Drive。已有attempt或科学残留时Run all只读，跳过源码/安装/模型，不补失败调用。
代理仅必要CPU工程/静态检查；真实E6/FFT6和完整Run all仍待用户。无摘要/版本/Git-clean/GPU型号或资源硬上限。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, subprocess
OUTPUT=Path('/content/drive/MyDrive/Video-WM/B-Real-Four-Phase-V1/fixed-seen-prefix-completion-v1')
def execution_artifacts(output):
    output=Path(output)
    names=('attempt.json','protocol.json','ledger.json','checkpoint.json','counts.json','numeric_completion.json','result.json','evaluation.json')
    found=[n+s for n in names for s in ('','.tmp') if (output/(n+s)).exists() or (output/(n+s)).is_symlink()]
    for folder in ('blind','features','preparation','references','evaluated'):
        root=output/folder
        if root.is_dir():
            found.extend(str(p.relative_to(output)) for p in root.rglob('*') if p.is_file() or p.is_symlink())
    return sorted(found)
EXISTING_ARTIFACTS=execution_artifacts(OUTPUT)
if not EXISTING_ARTIFACTS:
    SOURCE_ROOT=Path('/content/SC-SSTW-b-real-four-phase-v1')
    if not SOURCE_ROOT.exists():
        subprocess.run(['git','clone','--depth','1','--single-branch','--branch',
                        'dev/local-joint-state-payload-v1','https://github.com/RICHAAARC/SC-SSTW.git',
                        str(SOURCE_ROOT)],check=True)
    if not (SOURCE_ROOT/'experiments/wan_state_clock/b_real_four_phase_v1_run.py').is_file():
        raise FileNotFoundError('可用源码目录中缺少Card89运行入口，请使用包含该入口的源码目录：'+str(SOURCE_ROOT))
    OUTPUT.mkdir(parents=True,exist_ok=True)
    print('Available source',SOURCE_ROOT)
else:
    print('Existing attempt: saved records only; no source/setup/model reentry')


沿已成功Wan加载与导入修复路径执行；不会安装或运行媒体编解码工具。数据sidecar仅按成员读取，post成员在numeric marker后才读取。

In [ ]:

def report_progress(output):
    # Drive visibility can change between a file check and open. Progress is
    # informational; its read failure must not abandon or relaunch the worker.
    try:
        ledger = json.loads((output/"ledger.json").read_text())
        returned = sum(row.get("status") == "RETURNED" for row in ledger.values())
        active = [name for name, row in ledger.items() if row.get("status") == "STARTED"]
    except (OSError, ValueError, TypeError, AttributeError) as error:
        print("Progress temporarily unavailable; observing the same worker:",
              type(error).__name__, str(error), flush=True)
        return
    print(f"Completed core units: {returned}/{len(ledger)}; active: {active}", flush=True)


import subprocess, sys

EXISTING_ARTIFACTS = execution_artifacts(OUTPUT)
if not EXISTING_ARTIFACTS:
    LOG = OUTPUT/"setup.log"
    def command(args, label, check=True):
        print(label, flush=True)
        with LOG.open('a') as stream:
            stream.write('COMMAND '+repr(args)+'\n'); stream.flush()
            completed = subprocess.run(args, stdout=stream, stderr=subprocess.STDOUT)
        if completed.returncode and check:
            raise RuntimeError(label+' failed; see '+str(LOG))
        return completed.returncode

    # Copied from the already-used A/P2 setup; no A import/build dependency.
    def ensure_imports(probe, packages, label):
        import re
        module_packages = {package.replace('-', '_'): package for package in packages}
        repaired = set()
        while True:
            offset = LOG.stat().st_size if LOG.exists() else 0
            if not command([sys.executable, '-c', probe], ('Recheck ' if repaired else 'Probe ')+label, check=False):
                return
            with LOG.open('rb') as stream:
                stream.seek(offset); failure = stream.read().decode('utf-8', errors='replace')
            missing = re.search("ModuleNotFoundError: No module named ['\"]([^'\"]+)['\"]", failure)
            if "cannot import name 'FqnToConfig'" in failure and 'torchao.quantization' in failure:
                action = 'remove incompatible optional torchao'
                args = [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao']
            elif missing and missing.group(1) in module_packages:
                package = module_packages[missing.group(1)]
                action = 'install missing '+package
                args = [sys.executable, '-m', 'pip', 'install', package]
            else:
                raise RuntimeError(label+' failed without a known targeted repair; see '+str(LOG))
            if action in repaired:
                raise RuntimeError(label+' still fails after '+action+'; see '+str(LOG))
            repaired.add(action)
            command(args, 'Repair '+label+': '+action)

    ensure_imports('import numpy,torch', ['numpy', 'torch'], 'tensor dependencies')
    ensure_imports('import accelerate,safetensors,huggingface_hub; from diffusers import AutoencoderKLWan',
                   ['diffusers', 'accelerate', 'safetensors', 'huggingface-hub'], 'Wan VAE imports')
    with (OUTPUT/"runner.log").open("w") as log:
        process = subprocess.Popen([sys.executable, "-u", "-m",
            "experiments.wan_state_clock.b_real_four_phase_v1_run"],
            cwd=SOURCE_ROOT, stdout=log, stderr=subprocess.STDOUT)
    import time
    try:
        while process.poll() is None:
            time.sleep(30)
            report_progress(OUTPUT)
    except KeyboardInterrupt:
        process.terminate()
        try:
            process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        print("Stopped; saved STARTED calls remain unresolved and will not be replayed.")
        raise
    print("Runner return code:", process.returncode)
else:
    print("Existing attempt: displaying saved records; no physical calls repeated.")
print("Output:", OUTPUT)


保存结果：实际E/FFT与逻辑复用分开，源cell/段组/消息/undefined和假覆盖分开；最高分不是认证。

In [ ]:

saved={}
for name in ('result.json','checkpoint.json','ledger.json','numeric_completion.json','evaluation.json'):
    path=OUTPUT/name
    if path.is_file():
        try:saved[name]=json.loads(path.read_text())
        except (OSError,ValueError) as error:print('Unreadable saved file',name,type(error).__name__,str(error))
result=saved.get('result.json',saved.get('checkpoint.json',{}))
evaluation=saved.get('evaluation.json',{})
print('Execution',result.get('status'),'Actual new observation',result.get('actual_new_observation'))
print('New CPU',result.get('actual_new_CPU'),'Fixed denominators',result.get('fixed_denominator'))
for name,table in evaluation.get('tables',{}).items():
    print(name,'Family',table.get('family_status'),'Incomplete phases',table.get('incomplete_phases'))
    for variant,peak in table.get('variants',{}).items():
        row=peak.get('summary') or {}
        print(variant,peak.get('status'),'exact ties',peak.get('exact_max_ties'),
              'conditional phase/origin',row.get('phase'),row.get('public_origin_hypothesis'),
              'exact cells',row.get('exact_correct_cells'),'mixed unsupported',row.get('mixed_unsupported_single_cell_claims'),
              'group correct/wrong/undefined',row.get('group_correct_rows'),row.get('group_wrong_rows'),row.get('group_undefined_rows'),
              'present readable/errors',row.get('readable_present_bits'),row.get('present_errors'),
              'absent/partial source claims',row.get('absent_source_claims'),row.get('partial_source_claims'),
              'false missing-group bits',row.get('missing_group_false_coverage'))
for name,row in saved.get('ledger.json',{}).items():
    if row.get('status')!='RETURNED':print('Retained failed/interrupted/unentered',name,row)
print('No threshold, identity/phase certificate, FPR or trusted full message. Reuse is not new physical capacity.')
print('Full phase-specific support/undefined/source/message evidence: evaluation.json. Output',OUTPUT)
